# 🧜 Mermaid Diagrams: Swimlanes, Ordering & Styling

**A hands-on tutorial** — flowchart syntax, swimlane ordering, node styling, and layout control.

## What you'll learn
- Flowchart node shapes and edge types
- Swimlanes with `subgraph` — creation and ordering control
- The truth about invisible links (`~~~`) and what actually works
- Styling nodes with `classDef`, `class`, `style`, and `:::` shorthand
- The `%%{init}` block: themes, `themeVariables`, and `flowchart` config
- A complete real-world worked example and challenges

## Where Mermaid renders natively

| Environment | Syntax |
|---|---|
| GitHub / GitLab Markdown | ` ```mermaid ``` ` code fences |
| MkDocs (with plugin) | ` ```mermaid ``` ` code fences |
| Notion | `/mermaid` block |
| VS Code (with extension) | ` ```mermaid ``` ` preview |
| This notebook | `mmd()` helper below — **run Setup cell first** |

---

## Setup — Run This First

The `mmd()` helper renders diagrams inline using the Mermaid CDN.
Requires an internet connection. Run this cell before any diagram cell.

In [ ]:
from IPython.display import HTML, display
import uuid

def mmd(diagram: str):
    """Render a Mermaid diagram inline in Jupyter via CDN."""
    uid = 'm' + uuid.uuid4().hex[:8]
    html = (
        f"<div id='{uid}' style='background:#fff;padding:16px;"
        f"border-radius:8px;margin:8px 0;border:1px solid #e2e8f0;'>"
        f"<pre class='mermaid' style='margin:0;'>{diagram}</pre></div>"
        f"<script type='module'>"
        f"import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';"
        f"mermaid.initialize({{startOnLoad:false,securityLevel:'loose'}});"
        f"await mermaid.run({{nodes:[document.getElementById('{uid}').querySelector('.mermaid')]}});"
        f"</script>"
    )
    return HTML(html)

print('✅ mmd() ready — run this cell before any diagram cell.')

---
## 1. Flowchart Anatomy

A Mermaid flowchart has three parts:

```
%%{init: { "theme": "base", "flowchart": { "nodeSpacing": 40 } }}%%   ← optional
flowchart LR                                                             ← direction
    A[Node A] --> B[Node B]                                              ← nodes + edges
```

**Directions:** `LR` (left→right), `TD` (top→down), `RL`, `BT`

### Node Shape Reference

| Syntax | Shape | Use for |
|---|---|---|
| `ID[text]` | Rectangle | Process / step |
| `ID(text)` | Rounded rectangle | Soft step |
| `ID([text])` | Stadium | Start / end |
| `ID{text}` | Diamond | Decision |
| `ID[/text/]` | Right parallelogram | Input / document |
| `ID[\text\]` | Left parallelogram | Output |
| `ID[(text)]` | Cylinder | Database |
| `ID((text))` | Circle | Event |

### Edge Type Reference

| Syntax | Appearance | Use for |
|---|---|---|
| `A --> B` | Solid arrow | Normal flow |
| `A --- B` | Solid line, no arrow | Association |
| `A -.-> B` | Dashed arrow | Optional / async |
| `A ==> B` | Thick arrow | Critical path |
| `A -- label --> B` | Labelled arrow | Conditional branch |
| `A ~~~  B` | **Invisible** | Layout hint (see Section 4) |

Edge labels can also be written as `A -->|label| B`.

In [ ]:
# All common shapes in one diagram
mmd("""
flowchart LR
    A([Stadium: start/end])
    B[Rectangle: process]
    C(Rounded: soft step)
    D{Diamond: decision}
    E[/Parallelogram: doc/]
    F[(Cylinder: database)]

    A --> B --> C --> D -- yes --> E
    D -- no --> F
""")

---
## 2. Swimlanes with `subgraph`

A `subgraph` groups nodes into a labelled box — multiple subgraphs create swimlanes.

```
subgraph ID["Display Label"]
  direction LR          ← overrides layout direction inside this lane
  node1[...]
  node2[...]
end
```

- `ID` is used to reference the group in cross-lane edges
- The display label supports emoji: `"🧑 Worker"`
- `direction` inside a subgraph is **independent** of the outer diagram direction
- Cross-subgraph edges connect lanes: `Worker_node --> Client_node`

In [ ]:
# Basic two-lane swimlane
mmd("""
%%{init: {"theme": "base", "themeVariables": {"fontSize": "16px"}}}%%
flowchart LR

subgraph WORKER["🧑 Worker"]
  direction LR
  W1([Start]) --> W2[Do Work] --> W3([Done])
end

subgraph CLIENT["🏢 Client"]
  direction LR
  C1[Request] --> C2[Review] --> C3[/Pay/]
end

%% Cross-lane edges
C1 --> W1
W3 --> C3
""")

---
## 3. Controlling Swimlane Order

**The rule:** Mermaid renders swimlanes in **declaration order** — the order `subgraph` blocks appear in source.

- `flowchart LR` → subgraphs stack **top to bottom** in source order
- `flowchart TD` → subgraphs stack **left to right** in source order

**To reorder lanes: reorder the `subgraph` blocks.** That's it.

Cross-lane edges can sometimes fight declaration order when the layout engine infers rank
from the edge direction. If this happens, try reordering the edges themselves, or see
Section 4 on invisible link hints.

In [ ]:
# ❌ Client declared first → appears on top (may not be what you want)
mmd("""
%%{init: {"theme": "forest", "themeVariables": {"fontSize": "15px"}}}%%
flowchart LR

subgraph CLIENT["🏢 Client  ← declared first = top lane"]
  C1[Request] --> C2[Pay]
end

subgraph WORKER["🧑 Worker  ← declared second = bottom lane"]
  W1[Do Work] --> W2[Done]
end

C1 --> W1
W2 --> C2
""")

In [ ]:
# ✅ Swap the subgraph blocks — Worker now on top, Client on bottom
mmd("""
%%{init: {"theme": "forest", "themeVariables": {"fontSize": "15px"}}}%%
flowchart LR

subgraph WORKER["🧑 Worker  ← declared first = top lane"]
  W1[Do Work] --> W2[Done]
end

subgraph CLIENT["🏢 Client  ← declared second = bottom lane"]
  C1[Request] --> C2[Pay]
end

C1 --> W1
W2 --> C2
""")

---
## 4. Invisible Links (`~~~`) — What Works and What Doesn't

`~~~` creates a zero-width invisible edge. The layout engine counts it when placing
nodes but renders nothing visually.

### What reliably works
- `NodeA ~~~ NodeB` — pull two nodes into the same rank within a lane
- `A ~~~ B ~~~ C` — chain rank hints

### What often doesn't work
- `SubgraphID ~~~ SubgraphID` — Mermaid does **not** reliably accept subgraph
  labels as edge targets for invisible links in most versions
- Using `~~~` to fight a strong `-->` that already determines rank — the real edge wins

### The honest answer

> If your lanes are in the wrong order, **reorder the `subgraph` blocks**.
> That is the only reliable control.
>
> `~~~` is useful for nudging individual **nodes** to align at the same rank
> within a lane — not for controlling the lane order itself.

### When `~~~` is genuinely useful

You have two nodes that should visually align (same horizontal rank in `LR`),
but no real edge connects them. The next two cells show the before/after.

In [ ]:
# Without ~~~ — B1 naturally aligns with wherever the layout puts it
mmd("""
flowchart LR

subgraph A["Lane A"]
  A1[Step 1] --> A2[Step 2] --> A3[Step 3]
end
subgraph B["Lane B"]
  B1[Start] --> B2[End]
end

%% Real cross-lane edge enters at A1
A1 --> B1
""")

In [ ]:
# With ~~~ nudge — B1 pulled to align with A2 instead of A1
mmd("""
flowchart LR

subgraph A["Lane A"]
  A1[Step 1] --> A2[Step 2] --> A3[Step 3]
end
subgraph B["Lane B"]
  B1[Start] --> B2[End]
end

A1 --> B1

%% Invisible nudge — pull B1 into the same rank as A2
A2 ~~~ B1
""")

---
## 5. Styling with `classDef`

Mermaid uses CSS-like class definitions assigned to nodes.

### Syntax
```
classDef className  fill:#color, stroke:#color, stroke-width:Npx, color:#color,
                    font-size:Npx, font-weight:bold, stroke-dasharray:5 5

class nodeId1,nodeId2  className
```

### Inline shorthand with `:::`
```
A[My Node]:::className
```

### Style a single node (no class needed)
```
style nodeId  fill:#fef3c7, stroke:#d97706, stroke-width:2px
```

### Supported `classDef` properties

| Property | Example | Effect |
|---|---|---|
| `fill` | `fill:#dbeafe` | Background colour |
| `stroke` | `stroke:#1d4ed8` | Border colour |
| `stroke-width` | `stroke-width:3px` | Border thickness |
| `stroke-dasharray` | `stroke-dasharray:5 5` | Dashed border |
| `color` | `color:#111` | Text colour |
| `font-size` | `font-size:14px` | Text size |
| `font-weight` | `font-weight:bold` | Text weight |
| `rx` | `rx:12` | Corner radius |
| `opacity` | `opacity:0.6` | Transparency |

### Styling subgraph (lane) backgrounds
Global: `themeVariables.clusterBkg` in the init block.
Per-lane: `style SubgraphID fill:#color, stroke:#color`

In [ ]:
# classDef + class assignment
mmd("""
%%{init: {"theme": "base", "themeVariables": {"fontSize": "16px"}}}%%
flowchart LR

A([Start]):::start
--> B[Process]:::step
--> C{Decision}:::decision

C -- yes --> D[/Invoice/]:::doc
C -- no  --> E([End]):::start

D --> F[/Receipt/]:::payment

classDef start    fill:#dcfce7,stroke:#16a34a,stroke-width:3px,font-weight:bold
classDef step     fill:#dbeafe,stroke:#1d4ed8,stroke-width:2px
classDef decision fill:#fef9c3,stroke:#ca8a04,stroke-width:2px
classDef doc      fill:#fed7aa,stroke:#ea580c,stroke-width:2px
classDef payment  fill:#dcfce7,stroke:#16a34a,stroke-width:4px,font-weight:bold
""")

In [ ]:
# ::: inline shorthand + style one-off for a single node
mmd("""
flowchart LR

A[Approved]:::green --> B[Pending]:::yellow --> C[Rejected]:::red

classDef green  fill:#dcfce7,stroke:#16a34a,stroke-width:2px
classDef yellow fill:#fef9c3,stroke:#ca8a04,stroke-width:2px
classDef red    fill:#fee2e2,stroke:#dc2626,stroke-width:2px

%% One-off dashed border on a single node — no class needed
style B stroke-dasharray:6 3
""")

---
## 6. The `%%{init}` Block — Full Reference

The init block **must be the first line** of the diagram.

```
%%{init: {
  "theme": "base",
  "themeVariables": { ... },
  "flowchart": { ... }
}}%%
```

### Theme values

| Value | Description |
|---|---|
| `"default"` | Light, blue accents |
| `"base"` | Minimal — best for full `themeVariables` customisation |
| `"dark"` | Dark background |
| `"forest"` | Green-toned |
| `"neutral"` | Grayscale |

> Use `"base"` whenever you set `themeVariables`. Other themes apply their own
> values on top and may override yours — especially fill colours in `classDef`.

### `themeVariables` keys

| Key | Effect |
|---|---|
| `fontSize` | Global font size: `"18px"` |
| `fontFamily` | Font stack: `"Arial, Helvetica, sans-serif"` |
| `primaryColor` | Default node fill |
| `primaryTextColor` | Default node text |
| `primaryBorderColor` | Default node border |
| `lineColor` | Edge colour |
| `background` | Diagram background |
| `clusterBkg` | Subgraph (lane) background |
| `clusterBorder` | Subgraph border colour |
| `edgeLabelBackground` | Background behind edge labels |

### `flowchart` config keys

| Key | Values / Effect |
|---|---|
| `nodeSpacing` | Integer px — horizontal gap between nodes |
| `rankSpacing` | Integer px — vertical gap between ranks |
| `curve` | `"linear"` `"basis"` `"cardinal"` `"monotoneX"` |
| `padding` | Integer px — padding inside subgraph boxes |
| `useMaxWidth` | `true`/`false` — fit to container width |
| `htmlLabels` | `true` — allow HTML in node label text |

In [ ]:
# Change THEME to compare: "default" | "base" | "dark" | "forest" | "neutral"
THEME = "dark"

mmd(f"""
%%{{init: {{"theme": "{THEME}", "themeVariables": {{"fontSize": "16px"}}}}}}%%
flowchart LR
  A([Start]) --> B[Process] --> C{{Decision}}
  C -- yes --> D[/Output/]
  C -- no  --> A
""")

In [ ]:
# Full init block — tune nodeSpacing, rankSpacing, and curve
mmd("""
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "17px",
    "fontFamily": "Arial, Helvetica, sans-serif",
    "clusterBkg": "#f0f9ff",
    "clusterBorder": "#0284c7",
    "lineColor": "#475569",
    "edgeLabelBackground": "#f8fafc"
  },
  "flowchart": {
    "nodeSpacing": 50,
    "rankSpacing": 60,
    "curve": "basis",
    "padding": 20
  }
}}%%
flowchart LR

subgraph WORKER["🧑 Worker"]
  W1([Start]) --> W2[Do Work] --> W3([Done])
end
subgraph CLIENT["🏢 Client"]
  C1[Request] --> C2[Approved] --> C3[/Paid/]
end

C1 --> W1
W3 --> C3

classDef default fill:#dbeafe,stroke:#1d4ed8,stroke-width:2px
""")

---
## 7. Worked Example — Billing Workflow

This brings together everything covered:

- **Three swimlanes** — declaration order = visual order
- **Decision diamond** — conditional branching to different lanes
- **`classDef` classes** — doc / invoice / payment / status / decision types
- **Full `%%{init}` block** — `themeVariables` + `flowchart` tuning

### Business logic
1. Client sends a Statement of Work → Worker starts the job
2. Worker completes → decision: is this billable?
3. **yes** → client invoice flow activates; **no** → worker receives invoice directly
4. All documents captured in the Docs lane

In [ ]:
mmd("""
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "16px",
    "fontFamily": "Arial, Helvetica, sans-serif",
    "clusterBkg": "#f8fafc",
    "clusterBorder": "#94a3b8",
    "edgeLabelBackground": "#f1f5f9"
  },
  "flowchart": {
    "nodeSpacing": 40,
    "rankSpacing": 50,
    "curve": "linear"
  }
}}%%

flowchart LR

%% ===== SWIMLANES (top-to-bottom = declaration order) =====

subgraph JF["🧑 Worker"]
  direction LR
  JF_SOW([📄 Job Request])
  JF_DONE([WORK COMPLETED])
  JF_BILLABLE{billable?}
  JF_INV([🧾 INVOICE RECEIVED])
  JF_PAY[/✅ INVOICE PAID/]
end

subgraph BI["🏢 Client"]
  direction LR
  BI_INV([🧾 INVOICE SENT])
  BI_PAY[/✅ INVOICE PAID/]
end

subgraph CRE["📁 Documents"]
  direction TB
  SOW[🧾 SOW]
  Wk_INV[🧾 worker_INV]
  Cl_INV[🧾 client_INV]
  JF_RCPT[🧾 worker_RCPT]
  BI_RCPT[🧾 client_RCPT]
end

%% ===== WORKER FLOW =====
JF_SOW --> JF_DONE
JF_DONE --> JF_BILLABLE
JF_BILLABLE -- no  --> JF_INV
JF_BILLABLE -- yes --> BI_INV
JF_INV --> JF_PAY

%% ===== CLIENT FLOW =====
BI_INV --> BI_PAY

%% ===== DOCUMENT LINKS =====
SOW    --> JF_SOW
Wk_INV --> JF_INV
BI_INV --> Cl_INV
JF_PAY --> JF_RCPT
BI_PAY --> BI_RCPT

%% ===== STYLE =====
classDef doc      fill:#fef3c7,stroke:#d97706,stroke-width:2px
classDef invoice  fill:#fed7aa,stroke:#ea580c,stroke-width:2px
classDef payment  fill:#dcfce7,stroke:#16a34a,stroke-width:3px,font-weight:bold
classDef status   fill:#dbeafe,stroke:#1d4ed8,stroke-width:2px
classDef decision fill:#fef9c3,stroke:#ca8a04,stroke-width:2px

class SOW,Wk_INV,Cl_INV,JF_RCPT,BI_RCPT  doc
class BI_INV,JF_INV                        invoice
class BI_PAY,JF_PAY                        payment
class JF_DONE,JF_SOW                       status
class JF_BILLABLE                          decision
""")

---
## 8. Challenges

Work through these progressively — each builds on the last.

---
### Challenge 1 — Basic three-lane flowchart
Create a `flowchart LR` with three swimlanes: `Sales`, `Engineering`, `QA`.
Each lane has 2 nodes. Connect them in sequence across lanes.
Expected order top-to-bottom: Sales → Engineering → QA.

---
### Challenge 2 — Reverse the lane order
Take your Challenge 1 diagram and change the lane order to **QA → Engineering → Sales**
without changing any edge definitions. Only reorder the `subgraph` blocks.

---
### Challenge 3 — Add a decision loop
In the billing workflow above, after `WORK COMPLETED`, add a second decision:
`quality check passed?` — yes → proceed to `billable?`, no → loop back to job start.
Style the new decision yellow and label the fail edge `redo`.

---
### Challenge 4 — Theme comparison
Run the billing workflow with four different `"theme"` values:
`"default"` → `"dark"` → `"forest"` → `"neutral"`.
Note which themes override your `classDef` fill colours.
*(Hint: only `"base"` fully respects `classDef` fill colours.)*

---
### Challenge 5 — Init block tuning
In the billing workflow, experiment with:
- `nodeSpacing`: try `20`, `40`, `80` — cramped vs. spacious?
- `curve`: compare `"linear"` vs `"basis"` vs `"cardinal"` — which suits process flows?
- Give each swimlane a distinct background using `style SubgraphID fill:#color`

---
### Challenge 6 — Style an edge
Add a red `no` edge on the `billable?` decision using `linkStyle`.
Syntax: `linkStyle N stroke:red,stroke-width:3px` where N is the 0-based index
of the edge in declaration order (count your `-->` edges from the top).

In [ ]:
# Scratch cell — write your challenge solutions here
mmd("""
flowchart LR
  A[Your diagram here]
""")

---
## Quick Reference Card

| Goal | Solution |
|---|---|
| Control swimlane order | Declare `subgraph` blocks in the order you want |
| Force node rank alignment | `NodeA ~~~ NodeB` (nodes only — not subgraph IDs) |
| Style a group of nodes | `classDef name props` + `class nodeId name` |
| Style one node | `style nodeId fill:#f00,stroke:#900` |
| Inline style shorthand | `A[text]:::className` |
| Override lane direction | `direction LR` inside the `subgraph` block |
| Dashed edge | `A -.-> B` |
| Thick edge | `A ==> B` |
| No-arrow line | `A --- B` |
| Labelled edge (two forms) | `A -- label --> B` or `A -->|label| B` |
| Decision node | `ID{text}` |
| Best theme for custom colours | `"theme": "base"` |
| Subgraph background | `themeVariables.clusterBkg` (global) or `style ID fill:#color` |
| Edge curve style | `"flowchart": {"curve": "linear"|"basis"|"cardinal"}` |
| Style edge by index | `linkStyle N stroke:red,stroke-width:2px` |
| Invisible rank hint | `A ~~~ B` — limited; prefer reordering subgraphs |